# host

> What an application must supply to host the agent.

`Host` is the entire dependency of `leela.agent` on the world. Read it as the answer to
"what would I have to build to run this harness somewhere that is not leela?" -- the
answer is one class with eighteen methods, most of them one-liners over things any editor
already has.

It is written as a plain class rather than a `typing.Protocol` because the docstrings are
the specification. A `Protocol` would type-check the shape and document nothing, and the
shape is the easy part: what matters is that `check` really refuses paths outside the
open folders, and that `run_python` really cannot rebind the user's variables. A host
that satisfies the signatures and not the contracts is a host that quietly hands an agent
the whole filesystem.

`NullHost` implements the contract as "nothing is available", which is what makes the
harness testable without an IDE and what a partially-built host can inherit from.


In [ ]:
#| default_exp host

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
class Hit:
    "One search result, in the shape every backend of `Host.search` returns."
    def __init__(self, path, line=1, symbol='', text=''):
        self.path, self.line, self.symbol, self.text = path, line, symbol, text
    def __repr__(self): return f'{self.path}:{self.line}  {self.symbol}  {self.text}'

In [ ]:
#| export
class Host:
    """The application under an agent.

    Every method may raise; the tools in `tools.py` catch and report rather than let an
    exception end a turn. Methods that cannot be supported should raise
    `NotImplementedError`, which the tool list reads as "do not offer this tool" -- an
    agent told about a tool that always fails is worse off than one never told about it.
    """

    # -- where it is allowed to be -------------------------------------------
    @property
    def roots(self):
        "The open folders, as absolute paths. The agent is told about these and confined to them."
        raise NotImplementedError

    def check(self, path, must_exist=False):
        """Resolve `path` and refuse anything outside `roots`, returning a `Path`.

        This is the single chokepoint. exhash and fossick both write to disk on their own
        account, so each is handed a path this has already approved rather than a path the
        model supplied. Every other method here may assume its argument came through here.
        """
        raise NotImplementedError

    def walk(self):
        "Every readable file under the open folders."
        raise NotImplementedError

    def read(self, path):
        "One file's text, or None when it cannot be read."
        raise NotImplementedError

    def write(self, path, text):
        "Write `text` to `path`, through the same sandbox `check` enforces. Returns the path written."
        raise NotImplementedError

    def text_at(self, path):
        """One file as a single diffable document, `''` when it does not exist yet, None on error.

        Distinct from `read` because a notebook diffs as cell sources rather than as
        nbformat JSON, and because a file the agent is about to create must diff as a pure
        addition instead of as an error. This is what `Agent.changes()` compares.
        """
        raise NotImplementedError

    # -- seeing the code -----------------------------------------------------
    def search(self, query, limit=20):
        "Search the code index for `query`, returning `Hit`s. Semantic if an index exists, literal if not."
        raise NotImplementedError

    def peers(self, path, line, limit=20):
        "Code shaped like whatever is defined at `path`:`line` -- every place a pattern was already used."
        raise NotImplementedError

    def symbols(self, path):
        "The defs and classes in one file, as `Hit`s whose `score` is the indent depth."
        raise NotImplementedError

    @property
    def search_note(self):
        "Which engine answered, and anything it wants to say about why. Shown when a search finds nothing."
        return ''

    # -- reading the web -----------------------------------------------------
    def web_search(self, query, n=20):
        "Search the web; returns objects with `.title` and `.url`."
        raise NotImplementedError

    def read_url(self, url):
        "One page as markdown; returns an object with `.text`, or None."
        raise NotImplementedError

    def research(self, query):
        "Search and read the top results into one cited digest. Slower than `web_search`."
        raise NotImplementedError

    @property
    def research_note(self): return ''

    # -- notebooks -----------------------------------------------------------
    # The harness deliberately does not own a notebook representation. exhash addresses
    # cells by path and id without one, so only the two operations that genuinely need to
    # know what a notebook *is* are delegated here.
    def nb_cells(self, path):
        "`[(id, cell_type, first_line)]` for one notebook."
        raise NotImplementedError

    def nb_add_cell(self, path, source, index=-1, cell_type='code'):
        "Insert a cell (-1 appends), creating the notebook if needed. Returns the new cell's id."
        raise NotImplementedError

    # -- the live session ----------------------------------------------------
    def run_python(self, code):
        """Run `code` in the user's live namespace under whatever restrictions the host imposes.

        The contract the agent is briefed on: read anything, bind results to new names,
        never rebind or delete the owner's. Enforcing it is the host's job -- the harness
        only promises to tell the model about it.
        """
        raise NotImplementedError

    def inspect_python(self, code, scope='isolated'):
        """Run `code` against the live namespace without touching what the user has.

        Two scopes, and the difference is the interpreter you get rather than the safety you
        get -- both protect the owner's variables, by different means:

        - `'isolated'` runs in an allowlist sandbox on a *copy*. Attribute reads and builtins
          work; most library method calls are refused. Nothing can reach the real namespace
          at all, which is why it is the default and why it needs no trust.
        - `'overlay'` runs the real interpreter against the real namespace under an AST
          policy: the agent may read anything and bind its own names, which persist in its
          own layer, and cannot delete, rebind in place, or mutate the owner's. `list(df.columns)`
          and `df.head().to_dict()` work here; in the sandbox they do not.

        A host may refuse `'overlay'` (see `scopes`) and fall back to isolated, which is what
        a locked-down deployment does. Under a concurrent kernel either scope runs *alongside*
        a busy cell rather than queueing behind it.
        """
        raise NotImplementedError

    @property
    def scopes(self):
        "The scopes `inspect_python` will actually honour, most trusted last."
        return ('isolated',)

    @property
    def kernel_kind(self):
        """What runs the live namespace, and whether it can execute concurrently.

        `'ipymini'` means an inspection can run while a cell is busy; anything else means
        it queues behind whatever the kernel is already doing. The agent is told which,
        because "read the dataframe" is good advice under one and a way to hang the session
        under the other.
        """
        return 'ipykernel'

    @property
    def concurrent(self): return self.kernel_kind == 'ipymini'

    def list_vars(self):
        "What is in the live namespace: name, type, and a short value, one per line."
        raise NotImplementedError

    def terminal_text(self, lines=200):
        "What the IDE's terminal has printed. Read-only: this shows what the user ran, it cannot run anything."
        raise NotImplementedError

    # -- the person ----------------------------------------------------------
    @property
    def approvals(self):
        "The `Approvals` this host uses to put a write in front of a person, or None to approve everything."
        return None

    def note(self, text):
        "Tell the user something out of band (a status line). Never blocks; a host may drop it."
        pass

In [ ]:
#| export
class NullHost(Host):
    "A host with nothing behind it: every capability absent, so the harness runs bare in a test."

    def __init__(self, roots=()): self._roots = [str(r) for r in roots]

    @property
    def roots(self): return self._roots

    def check(self, path, must_exist=False):
        from pathlib import Path
        return Path(path)

    def walk(self): return []
    def read(self, path): return None
    def write(self, path, text): raise NotImplementedError
    def text_at(self, path): return None
    def search(self, query, limit=20): return []
    def peers(self, path, line, limit=20): return []
    def symbols(self, path): return []
    def web_search(self, query, n=20): return []
    def read_url(self, url): return None
    def research(self, query): return ''
    def nb_cells(self, path): raise NotImplementedError
    def nb_add_cell(self, path, source, index=-1, cell_type='code'): raise NotImplementedError
    def run_python(self, code): raise NotImplementedError
    def inspect_python(self, code, scope='isolated'): raise NotImplementedError
    def list_vars(self): raise NotImplementedError
    def terminal_text(self, lines=200): raise NotImplementedError

## Tests


In [ ]:
# A host says "I cannot do that" by raising, and the harness reads that as
# "do not offer the tool". Nothing else in the package has to know why.
h = NullHost(['/proj'])
print('roots     :', h.roots)
print('read      :', h.read('/proj/a.py'))
print('scopes    :', h.scopes)
print('kernel    :', h.kernel_kind, '| concurrent:', h.concurrent)
assert h.roots == ['/proj']
assert h.scopes == ('isolated',) and not h.concurrent

In [ ]:
# The absent capabilities raise rather than return junk.
for name, call in [('write', lambda: h.write('a', 'x')),
                   ('run_python', lambda: h.run_python('1')),
                   ('list_vars', h.list_vars)]:
    try: call(); raise AssertionError(f'{name} should have raised')
    except NotImplementedError: print(f'{name:12} -> NotImplementedError (tool will not be offered)')

In [ ]:
# `Hit` is the one shape every search backend returns, so the tools never branch on engine.
hit = Hit('leela/ai.py', 240, 'WorkspaceHost', 'class WorkspaceHost(Host):')
print(hit)
assert (hit.path, hit.line, hit.symbol) == ('leela/ai.py', 240, 'WorkspaceHost')